## Análisis de Desempeño de Operadores – CallMeMaybe

El objetivo principal de este estudio es identificar operadores ineficaces a partir del análisis de métricas clave, tales como llamadas perdidas, duración de llamadas y tiempo de espera. Asimismo, se busca validar los resultados mediante pruebas estadísticas y proponer recomendaciones para mejorar el servicio.

Una vez identificado el problema de negocio, la intencion es poder desarrollar el proyecto en las siguientes etapas:

- Descripción y limpieza de los datos

- Análisis exploratorio (EDA)

- Definición de métricas e identificación de operadores ineficaces

- Pruebas de hipótesis

- Indicadores propuestos (KPIs)

- Conclusiones y recomendaciones


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

plt.style.use('default')
sns.set()

In [3]:
calls = pd.read_csv('/datasets/telecom_dataset.csv')
clients = pd.read_csv('/datasets/telecom_clients.csv')

In [4]:
print(calls.shape)
print(" ")
calls.info()
print(" ")

(53902, 9)
 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   user_id              53902 non-null  int64  
 1   date                 53902 non-null  object 
 2   direction            53902 non-null  object 
 3   internal             53785 non-null  object 
 4   operator_id          45730 non-null  float64
 5   is_missed_call       53902 non-null  bool   
 6   calls_count          53902 non-null  int64  
 7   call_duration        53902 non-null  int64  
 8   total_call_duration  53902 non-null  int64  
dtypes: bool(1), float64(1), int64(4), object(3)
memory usage: 3.3+ MB
 


Al revisar los datos encontramos lo siguiente:

El **DataFrame Calls** cuenta con 53.902 filas y 9 columnas. Cada fila representa información agregada sobre llamadas realizadas o recibidas por los usuarios y operadores del sistema.

**Valores faltantes.**
Se identificaron valores nulos en la variable:
operator_id: 8.172 valores faltantes (53.902 - 45.730)
Estos registros corresponden probablemente a llamadas que no fueron atendidas por un operador específico o a procesos automáticos del sistema.
La variable internal también presenta una pequeña cantidad de valores faltantes (117 registros).
El resto de las variables no presenta valores nulos.

**Registros duplicados**
No se identificaron registros duplicados.

**Tipos de datos**
El conjunto de datos presenta los siguientes tipos:
**Variables numéricas:** user_id, calls_count, call_duration, total_call_duration
**Variable booleana:** is_missed_call
**Variables categóricas:** direction, internal
**Identificador con decimales:** operator_id (float64)
**Variable temporal:** date (object)

Se observa que:
La variable date se encuentra en formato texto, por lo que debe ser convertida a formato datetime.
operator_id debería manejarse como variable categórica o entera, ya que representa un identificador. 
Estos cambios se realizarán a continuación.

In [5]:
# Limpieza de los datos

# Convertir fecha
calls['date'] = pd.to_datetime(calls['date'], errors='coerce')
calls = calls.dropna(subset=['date'])

# Arreglar internal
calls['internal'] = calls['internal'].fillna(False)
calls['internal'] = calls['internal'].astype(bool)

# Eliminar filas sin operador
calls = calls.dropna(subset=['operator_id'])

# Convertir operator_id a entero
calls['operator_id'] = calls['operator_id'].astype(int)

# Crear tiempo de espera
calls['waiting_time'] = (
    calls['total_call_duration'] - calls['call_duration']
)

# Eliminar tiempos negativos
calls = calls[calls['waiting_time'] >= 0]

In [6]:
calls.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 45730 entries, 1 to 53900
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype                                
---  ------               --------------  -----                                
 0   user_id              45730 non-null  int64                                
 1   date                 45730 non-null  datetime64[ns, pytz.FixedOffset(180)]
 2   direction            45730 non-null  object                               
 3   internal             45730 non-null  bool                                 
 4   operator_id          45730 non-null  int64                                
 5   is_missed_call       45730 non-null  bool                                 
 6   calls_count          45730 non-null  int64                                
 7   call_duration        45730 non-null  int64                                
 8   total_call_duration  45730 non-null  int64                                
 9   waitin

Se realizó un proceso de limpieza y transformación de los datos con el fin de garantizar su calidad para el análisis posterior. En primer lugar, se convirtió la variable de fecha al formato datetime y se eliminaron registros con valores inválidos.
Posteriormente, se trataron los valores faltantes en la variable internal y se eliminaron las filas sin operador asignado, conservando únicamente aquellas asociadas a un desempeño individual.
Asimismo, se corrigieron los tipos de datos, convirtiendo operator_id a formato entero y las variables categóricas a booleanas cuando correspondía.
Adicionalmente, se creó la variable waiting_time, calculada como la diferencia entre la duración total y la duración efectiva de las llamadas, y se eliminaron registros con valores inconsistentes.
Como resultado, el conjunto final quedó conformado por 45.730 registros sin valores faltantes y con tipos de datos adecuados.

In [7]:
#Resumen estadístico

print(calls.describe())

             user_id    operator_id   calls_count  call_duration  \
count   45730.000000   45730.000000  45730.000000   45730.000000   
mean   167302.220315  916535.993002     16.904417    1009.765121   
std       600.731240   21254.123136     59.045234    4014.600085   
min    166377.000000  879896.000000      1.000000       0.000000   
25%    166782.000000  900788.000000      1.000000       0.000000   
50%    167175.000000  913938.000000      4.000000     106.000000   
75%    167827.000000  937708.000000     14.000000     772.000000   
max    168606.000000  973286.000000   4817.000000  144395.000000   

       total_call_duration  waiting_time  
count         45730.000000  45730.000000  
mean           1322.141789    312.376667  
std            4732.427880   1174.185593  
min               0.000000      0.000000  
25%              68.000000     19.000000  
50%             290.500000     60.000000  
75%            1110.000000    220.000000  
max          166155.000000  46474.000000  


El análisis descriptivo muestra que el número de llamadas presenta una media de 16,9 y una mediana de 4, lo que indica una distribución asimétrica influenciada por valores extremos.
La duración promedio de las llamadas es de 1.009,8 segundos, con una mediana de 106 segundos, mientras que la duración total presenta una media de 1.322,1 segundos. En ambos casos se observa una alta variabilidad.
El tiempo de espera tiene un valor medio de 312,4 segundos y una mediana de 60 segundos, aunque existen casos con tiempos excesivos de hasta 46.474 segundos.
En general, la mayoría de los registros presenta valores moderados, pero se identifican valores atípicos que pueden afectar negativamente la experiencia del cliente, lo que justifica la necesidad de analizar la eficiencia de los operadores.

# Identificación de variables relevantes y operadores Ineficaces.

Las variables más relevantes son aquellas que permiten evaluar directamente el desempeño de los operadores y segmentar la información para su interpretación.

- is_missed_call: indica si una llamada fue perdida.
- calls_count: representa el número de llamadas gestionadas.
- call_duration: mide la duración de las llamadas en segundos
- total_call_duration: tiempo total dedicado a llamadas
- waiting_time: tiempo de espera de los clientes antes de ser atendidos
- direction: ermite distinguir entre llamadas entrantes y saliente

En cuanto a varables de agregacion.

- operator_id: variable principal para agrupar datos y evaluar el desempeño individual.
user_id: permite analizar patrones desde la perspectiva del cliente.
- date: facilita el análisis temporal, identificando tendencias diarias, mensuales o estacionales.

Se considera que un operador es ineficaz cuando presenta, de forma consistente, alguno de los siguientes comportamientos:

- Alta proporción de llamadas perdidas. (is_missed_call)
- Tiempos de espera elevados. (waiting_time)
- Bajo volumen de llamadas salientes, cuando corresponde. (direction)
  
Estos indicadores afectan negativamente la experiencia del cliente y la eficiencia operativa. A continuacion encontraremos operadores ineficaces.


In [8]:
# Datos por operador

operator_stats = calls.groupby('operator_id').agg(
    total_calls = ('calls_count', 'sum'),
    missed_calls = ('is_missed_call', 'sum'),
    avg_duration = ('call_duration', 'mean'),
    avg_waiting = ('waiting_time', 'mean'),
    outgoing_calls = ('direction', lambda x: (x == 'out').sum())
).reset_index()

# Tasas

operator_stats['missed_rate'] = (
    operator_stats['missed_calls'] / operator_stats['total_calls']
)

# Percentiles

missed_limit = operator_stats['missed_rate'].quantile(0.75)
wait_limit = operator_stats['avg_waiting'].quantile(0.75)
out_limit = operator_stats['outgoing_calls'].quantile(0.25)

# Clasificación de operadores

operator_stats['ineffective'] = (
    (operator_stats['missed_rate'] > missed_limit) |
    (operator_stats['avg_waiting'] > wait_limit) |
    (operator_stats['outgoing_calls'] < out_limit)
)

print(operator_stats['ineffective'].value_counts())
print(" ")
print(operator_stats[operator_stats['ineffective']].head())

True     748
False    344
Name: ineffective, dtype: int64
 
   operator_id  total_calls  missed_calls  avg_duration  avg_waiting  \
1       879898         7974           100   1111.067729   450.087649   
2       880020           54             7    104.090909    15.181818   
3       880022          219            33    240.842105    57.565789   
5       880028         2906            91    746.265487   210.592920   
6       880240           49             8    275.928571    40.928571   

   outgoing_calls  missed_rate  ineffective  
1             187     0.012541         True  
2              14     0.129630         True  
3              68     0.150685         True  
5             181     0.031315         True  
6              14     0.163265         True  


A partir de los criterios definidos, se clasificaron 1.092 operadores, de los cuales 748 (68%) fueron identificados como ineficaces y 344 (32%) como eficientes.

Los operadores ineficaces presentan, al menos, uno de los siguientes factores: alta tasa de llamadas perdidas, tiempos de espera elevados o bajo volumen de llamadas salientes. Se observa que las causas de ineficiencia varían entre operadores, lo que sugiere la necesidad de intervenciones diferenciadas.

Estos resultados permiten enfocar acciones de mejora en los operadores con mayor impacto negativo en la experiencia del cliente.

# Planteamiento de Hipótesis
A continuacion se formulan hipótesis estadísticas con el fin de evaluar si existen diferencias significativas entre los operadores eficientes e ineficaces, y determinar los factores que influyen en el bajo desempeño.

**Hipotesis 1.** 
Evaluar si los operadores ineficaces presentan una mayor proporción de llamadas perdidas.

**H₀:**
No existe diferencia significativa en la tasa de llamadas perdidas entre operadores eficientes e ineficaces.

**H₁:**
Los operadores ineficaces presentan una tasa de llamadas perdidas significativamente mayor que los operadores eficientes.

In [9]:
# Separar operadores eficientes e ineficientes

bad = operator_stats[operator_stats['ineffective']]
good = operator_stats[~operator_stats['ineffective']]

Antes de decidir usar t-test, miramos si los datos son normales, para las variables que vamos a utilizar.

In [10]:
from scipy.stats import shapiro

def test_normality(var, name):
    n_bad = min(500, len(bad))
    n_good = min(500, len(good))
    
    print(f"\n{name} - Operadores ineficaces:")
    print(shapiro(bad[var].sample(n_bad)))
    
    print(f"{name} - Operadores eficaces:")
    print(shapiro(good[var].sample(n_good)))

# Pruebas
test_normality('missed_rate', 'Missed Rate')
test_normality('avg_waiting', 'Tiempo de espera')
test_normality('outgoing_calls', 'Llamadas salientes')


Missed Rate - Operadores ineficaces:
ShapiroResult(statistic=0.6266975402832031, pvalue=1.6779869176490835e-31)
Missed Rate - Operadores eficaces:
ShapiroResult(statistic=0.9499052166938782, pvalue=2.0326167415873897e-09)

Tiempo de espera - Operadores ineficaces:
ShapiroResult(statistic=0.46255218982696533, pvalue=4.0443882568526334e-36)
Tiempo de espera - Operadores eficaces:
ShapiroResult(statistic=0.9154576063156128, pvalue=5.474800842709959e-13)

Llamadas salientes - Operadores ineficaces:
ShapiroResult(statistic=0.688228964805603, pvalue=2.426068522650804e-29)
Llamadas salientes - Operadores eficaces:
ShapiroResult(statistic=0.8044241070747375, pvalue=4.5329048434682694e-20)


La prueba de Shapiro-Wilk mostró p-values inferiores a 0.05 para todas las variables analizadas, lo que indica ausencia de normalidad y justifica el uso de pruebas no paramétricas.

In [11]:
from scipy.stats import mannwhitneyu

u1, p1 = mannwhitneyu(
    bad['missed_rate'],
    good['missed_rate'],
    alternative='greater'
)

alpha = 0.05

print(f"Estadístico U: {u1:.2f}")
print(f"Valor p: {p1:.4f}")

if p1 < alpha:
    print("Resultado: Se rechaza la hipótesis nula.")
else:
    print("Resultado: No se rechaza la hipótesis nula.")

Estadístico U: 116811.00
Valor p: 0.9930
Resultado: No se rechaza la hipótesis nula.


La prueba de Mann-Whitney U arrojó un valor p de 0.993, superior al nivel de significancia de 0.05. Por lo tanto, no se rechaza la hipótesis nula, indicando que no existe evidencia estadística suficiente para afirmar que los operadores ineficaces presentan una mayor tasa de llamadas perdidas.

**Hipótesis 2: Tiempo de espera**

Determinar si el tiempo de espera influye en la ineficiencia operativa.

**H₀:**
No existe diferencia significativa en el tiempo promedio de espera entre operadores eficientes e ineficaces.

**H₁:**
Los operadores ineficaces presentan un tiempo de espera promedio significativamente mayor.

In [12]:
u2, p2 = mannwhitneyu(
    bad['avg_waiting'],
    good['avg_waiting'],
    alternative='greater'
)



alpha = 0.05

print(f"Estadístico U: {u2:.2f}")
print(f"Valor p: {p2:.4f}")

if p2 < alpha:
    print("Resultado: Se rechaza la hipótesis nula.")
else:
    print("Resultado: No se rechaza la hipótesis nula.")


Estadístico U: 134485.50
Valor p: 0.1143
Resultado: No se rechaza la hipótesis nula.


El valor p obtenido fue de 0.1143, superior al nivel de significancia de 0.05. Por lo tanto, no se rechaza la hipótesis nula, lo que indica que no existe evidencia estadística suficiente para afirmar que los operadores ineficaces presentan mayores tiempos de espera.

**Hipótesis 3: Llamadas salientes**

Evaluar si el bajo volumen de llamadas salientes está asociado a la ineficiencia.

**H₀:**
No existe diferencia significativa en el número de llamadas salientes entre operadores eficientes e ineficaces.

**H₁:**
Los operadores ineficaces realizan significativamente menos llamadas salientes.

In [13]:
u3, p3 = mannwhitneyu(
    bad['outgoing_calls'],
    good['outgoing_calls'],
    alternative='less'
)

alpha = 0.05

print(f"Estadístico U: {u3:.2f}")
print(f"Valor p: {p3:.4f}")

if p3 < alpha:
    print("Resultado: Se rechaza la hipótesis nula.")
else:
    print("Resultado: No se rechaza la hipótesis nula.")

Estadístico U: 85086.00
Valor p: 0.0000
Resultado: Se rechaza la hipótesis nula.


El valor p obtenido fue inferior a 0.001, menor que el nivel de significancia de 0.05. Por lo tanto, se rechaza la hipótesis nula, lo que indica que existe una diferencia estadísticamente significativa en la cantidad de llamadas salientes entre ambos grupos.


Debido a la ausencia de normalidad en las variables analizadas, se utilizó la prueba de Mann-Whitney U para contrastar las hipótesis planteadas.

# KPIs definidos para este proyecto

# KPI 1 – Proporción de operadores ineficaces

**¿Qué mide?**

El porcentaje de operadores clasificados como ineficaces frente al total de operadores.

**¿Cómo se define un operador ineficaz?**

Un operador se considera ineficaz cuando presenta:

- Alta cantidad de llamadas entrantes perdidas
- Tiempos de espera elevados para las llamadas entrantes

**¿Por qué es importante?**

Este KPI entrega una visión general del problema, permite dimensionar qué tan extendida está la ineficiencia y funciona como indicador de alerta temprana para la operación

**¿Cómo se presenta en el dashboard?**

Gráfico de barras, donde se muestra la diferenciación clara entre operadores eficaces vs ineficaces.


# KPI 2 – Tasa de llamadas perdidas (Missed Call Rate)

**¿Qué mide?**

La proporción de llamadas entrantes que no fueron atendidas respecto al total de llamadas.

**¿Por qué es importante?**

La tasa de llamadas perdidas impacta directamente en:

- Experiencia del cliente
- Nivel de servicio
- Potencial pérdida de ingresos

**¿Cómo se presenta en el dashboard?**

- Gráfico comparativo (barras o distribución)
- Línea de referencia con el promedio global, para facilitar la interpretación
- Permite identificar operadores o grupos por encima del nivel aceptable


# KPI 3 – Tiempo promedio de espera por operador

**¿Qué mide?**

El tiempo promedio que esperan los clientes antes de ser atendidos por cada operador.

**¿Por qué es importante?**

Este KPI permite:

- Identificar cuellos de botella operativos
- Detectar inconsistencias entre operadores
- Entender si la ineficiencia es estructural o individual

**¿Cómo se presenta en el dashboard?**

Se utiliza el tipo de grafico Scatter plot por operador, asi mismo se separó por color la eficacia, lo cual permite ver patrones, dispersion y outliers de forma clara.

In [14]:
# Dashboard

dashboard_data = calls.groupby('operator_id').agg(
    total_calls=('calls_count', 'sum'),
    missed_calls=('is_missed_call', 'sum'),
    outgoing_calls=('direction', lambda x: (x == 'out').sum()),
    avg_waiting=('waiting_time', 'mean'),
    avg_duration=('call_duration', 'mean')
).reset_index()

dashboard_data['missed_rate'] = (
    dashboard_data['missed_calls'] / dashboard_data['total_calls']
)

dashboard_data['ineffective'] = dashboard_data['operator_id'].isin(
    bad['operator_id']
)

dashboard_data.head()

,operator_id,total_calls,missed_calls,outgoing_calls,avg_waiting,avg_duration,missed_rate,ineffective
0,879896,1131,50,105,110.671875,650.476562,0.044209,False
1,879898,7974,100,187,450.087649,1111.067729,0.012541,True
2,880020,54,7,14,15.181818,104.090909,0.129630,True
3,880022,219,33,68,57.565789,240.842105,0.150685,True
4,880026,2439,94,179,121.171717,856.939394,0.038540,False


In [16]:
dashboard_data.to_csv('dashboard_data.csv', index=False)

# Dashboard Tableau.

En el siguien enlace encontrarás el Dashboard, donde se analizand los KPIs propuestos para este proyecto.

https://public.tableau.com/app/profile/victoria.diago/viz/Eficienciadeoperadores-CallMeMaybe/Dashboard1?publish=yes 


# Conclusiónes

A partir del análisis exploratorio, la limpieza de datos y las pruebas estadísticas realizadas, se identificaron patrones relevantes en el desempeño de los operadores del servicio CallMeMaybe.

En primer lugar, el análisis de normalidad mostró que las principales métricas evaluadas (missed_rate, avg_waiting y outgoing_calls) no siguen una distribución normal, lo que justificó el uso de pruebas no paramétricas para la evaluación de diferencias entre grupos.

Las pruebas de hipótesis indicaron que no existen diferencias estadísticamente significativas entre operadores eficaces e ineficaces en términos de llamadas perdidas ni en el tiempo promedio de espera. Por el contrario, se encontró una diferencia estadísticamente significativa en el número de llamadas salientes entre ambos grupos, lo que indica que esta variable está directamente asociada con la ineficacia operativa. 

Desde una perspectiva de negocio, estos resultados sugieren que la empresa debería priorizar estrategias orientadas a mejorar la gestión de llamadas salientes, tales como programas de capacitación, redistribución de carga laboral y monitoreo continuo del desempeño.

Finalmente, este análisis proporciona una base objetiva para el diseño de políticas de evaluación y mejora del desempeño, permitiendo a CallMeMaybe optimizar sus recursos humanos, reducir ineficiencias y mejorar la calidad del servicio al cliente.